# UMLS-first MIMIC-IV discharge property graph vLLM test grid

This notebook builds LlamaIndex property graphs from a small MIMIC-IV discharge-note subset and tests vLLM-served open-weight model combinations.

vLLM is used through its OpenAI-compatible API. Start one or more models behind `VLLM_BASE_URL` before running the grid. The notebook discovers served model IDs from `/models` and filters the candidate sweep to models that are actually available.

For embeddings, use either a vLLM embedding endpoint or set `INDEX_EMBEDDING_PROVIDER` to another supported backend before running the setup cell.


In [ ]:
from __future__ import annotations

import asyncio
import os
import sys
from pathlib import Path

import pandas as pd
import requests

repo_root = Path.cwd()
if not (repo_root / "main.py").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "main.py").exists():
            repo_root = parent
            break

sys.path.insert(0, str(repo_root))

os.environ.setdefault("INDEX_LLM_PROVIDER", "vllm")
os.environ.setdefault("INDEX_EMBEDDING_PROVIDER", "vllm")
os.environ.setdefault("VLLM_BASE_URL", "http://127.0.0.1:8000/v1")
os.environ.setdefault("VLLM_API_KEY", "EMPTY")
os.environ.setdefault("INDEX_LLM_REQUEST_TIMEOUT", "240")
os.environ.setdefault("MPLCONFIGDIR", str(repo_root / "output" / ".matplotlib"))

from eval.medqa_smoke import load_questions, format_options, extract_answer
from ingest.mimic import MimicDischargeSubsetConfig, extract_mimic_discharge_subset
from rag.index import ensure_index
from rag.retrieve import query_index_context
from llama_index.llms.openai import OpenAI
from rag.visualize import save_clinical_entity_graph, save_clinical_entity_graph_jpeg

# Popular vLLM-friendly open-weight instruct models. Availability depends on
# what your vLLM server is currently serving and what your Hugging Face token can access.
candidate_generation_models = [
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen3-8B",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "meta-llama/Llama-3.1-8B-Instruct",
    "google/gemma-2-9b-it",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
]

candidate_embedding_models = [
    "BAAI/bge-small-en-v1.5",
    "BAAI/bge-base-en-v1.5",
    "intfloat/e5-small-v2",
]

def model_slug(value: object | None) -> str:
    text = "unknown" if value is None else str(value)
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in text).strip("_") or "unknown"

def combo_slug(generation_model: str, embedding_model: str) -> str:
    return f"gen-{model_slug(generation_model)}__embed-{model_slug(embedding_model)}"

def env_csv(name: str) -> list[str]:
    value = os.environ.get(name, "").strip()
    return [item.strip() for item in value.split(",") if item.strip()]

def served_vllm_models(base_url: str) -> list[str]:
    try:
        response = requests.get(f"{base_url.rstrip('/')}/models", timeout=10)
        response.raise_for_status()
    except Exception as exc:
        print(f"Could not read vLLM /models endpoint: {exc}")
        return []
    payload = response.json()
    return [item.get("id") for item in payload.get("data", []) if item.get("id")]

vllm_base_url = os.environ["VLLM_BASE_URL"]
served_models = served_vllm_models(vllm_base_url)
requested_generation_models = env_csv("VLLM_MODEL_SWEEP") or candidate_generation_models
generation_model_sweep = [model for model in requested_generation_models if not served_models or model in served_models]
if not generation_model_sweep:
    fallback_model = os.environ.get("VLLM_MODEL") or (served_models[0] if served_models else candidate_generation_models[0])
    generation_model_sweep = [fallback_model]

requested_embedding_models = env_csv("VLLM_EMBEDDING_MODEL_SWEEP") or [os.environ.get("VLLM_EMBEDDING_MODEL", candidate_embedding_models[0])]
embedding_model_sweep = requested_embedding_models

print("repo_root:", repo_root)
print("python:", sys.version.split()[0])
print("vllm_base_url:", vllm_base_url)
print("served_models:", served_models or "unknown")
print("index_llm_provider:", os.environ.get("INDEX_LLM_PROVIDER"))
print("index_embedding_provider:", os.environ.get("INDEX_EMBEDDING_PROVIDER"))
print("generation_model_sweep:", generation_model_sweep)
print("embedding_model_sweep:", embedding_model_sweep)
print("total_combinations:", len(generation_model_sweep) * len(embedding_model_sweep))


In [ ]:
# Source paths.
mimic_csv = repo_root / "data" / "mimic_iv_note" / "discharge.csv"
mimic_subset_dir = repo_root / "data" / "evidence" / "mimic_discharge_subset"
subset_limit = None
subset_max_chars = 3000
test_jsonl_candidates = [
    repo_root / "test.jsonl",
    repo_root / "data" / "medqa" / "data_clean" / "questions" / "US" / "test.jsonl",
    repo_root / "data" / "eval" / "test.jsonl",
]
test_jsonl = next((path for path in test_jsonl_candidates if path.exists()), None)
output_dir = repo_root / "output" / "vllm"
output_dir.mkdir(parents=True, exist_ok=True)

use_umls = True
schema_guided = False

print("mimic_csv:", mimic_csv)
print("mimic_subset_dir:", mimic_subset_dir)
print("test_jsonl:", test_jsonl)
print("use_umls:", use_umls)
print("schema_guided:", schema_guided)


In [ ]:
# Extract all matching discharge notes used by every model combination.
if not mimic_csv.exists():
    raise FileNotFoundError(f"Missing MIMIC discharge CSV: {mimic_csv}")

extract_mimic_discharge_subset(
    MimicDischargeSubsetConfig(
        csv_path=mimic_csv,
        output_dir=mimic_subset_dir,
        limit=subset_limit,
        note_type="DS",
        max_chars=subset_max_chars,
        overwrite=True,
    )
)
print("Prepared MIMIC subset in:", mimic_subset_dir)


In [ ]:
# Load a small test set.
if test_jsonl is not None:
    total_questions = sum(1 for line in test_jsonl.open("r", encoding="utf-8") if line.strip())
    sample_size = min(15, total_questions)
    questions = load_questions(test_jsonl, sample_size=sample_size)
else:
    questions = [
        {
            "id": f"demo-{i}",
            "question": "What is the preferred next step for a patient with CKD and uncontrolled hypertension?",
            "answer": "A",
            "options": {
                "A": "Optimize blood pressure control and renal protection",
                "B": "Stop all medication",
                "C": "Ignore the blood pressure",
            },
        }
        for i in range(1, 11)
    ]

pd.DataFrame([{"id": q.get("id"), "question": q.get("question"), "answer": q.get("answer")} for q in questions])


In [ ]:
# Build/query each generation-model x embedding-model combination and collect artifacts.
rows = []
artifact_rows = []
artifact_dir = output_dir / "vllm_property_graph_artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
vllm_base_url = os.environ.get("VLLM_BASE_URL", "http://127.0.0.1:8000/v1")

for generation_model in generation_model_sweep:
    os.environ["INDEX_LLM_MODEL"] = generation_model
    os.environ["VLLM_MODEL"] = generation_model
    for embedding_model in embedding_model_sweep:
        os.environ["INDEX_EMBEDDING_MODEL"] = embedding_model
        os.environ["VLLM_EMBEDDING_MODEL"] = embedding_model
        slug = combo_slug(generation_model, embedding_model)
        indexed_graph = await asyncio.to_thread(
            ensure_index,
            input_dir=mimic_subset_dir,
            output_dir=output_dir,
            use_umls=use_umls,
            schema_guided=schema_guided,
        )

        query_llm = OpenAI(
            model=generation_model,
            api_key=os.environ.get("VLLM_API_KEY", "EMPTY"),
            api_base=vllm_base_url,
            timeout=float(os.environ.get("INDEX_LLM_REQUEST_TIMEOUT", "240")),
        )
        html_path = artifact_dir / f"{slug}.html"
        jpeg_path = artifact_dir / f"{slug}.jpg"
        results_path = artifact_dir / f"{slug}_results.csv"
        graph_html = save_clinical_entity_graph(output_dir, html_path, source="auto")
        graph_jpeg = save_clinical_entity_graph_jpeg(
            output_dir,
            jpeg_path,
            source="auto",
            title=f"Property graph: {slug}",
        )
        print("Saved graph artifacts:", graph_html, graph_jpeg)

        combo_rows = []
        for item in questions:
            prompt = item["question"]
            if item.get("options"):
                prompt = f"{prompt}\n\n" + format_options(item["options"])
            response, context = await query_index_context(index=indexed_graph, query=prompt, llm=query_llm)
            predicted = extract_answer(response, item["options"]) if item.get("options") else None
            combo_rows.append({
                "id": item.get("id"),
                "question": item["question"],
                "gold_answer": item.get("answer"),
                "predicted": predicted,
                "model": generation_model,
                "embedding_model": embedding_model,
                "combo_slug": slug,
                "response": response[:1000],
                "context_preview": context[:1000] if context else None,
                "html_plot": graph_html.as_posix(),
                "jpeg_plot": graph_jpeg.as_posix(),
            })

        combo_results = pd.DataFrame(combo_rows)
        combo_results.to_csv(results_path, index=False)
        rows.extend(combo_rows)
        artifact_rows.append({
            "combo_slug": slug,
            "model": generation_model,
            "embedding_model": embedding_model,
            "html_plot": graph_html.as_posix(),
            "jpeg_plot": graph_jpeg.as_posix(),
            "results_csv": results_path.as_posix(),
            "question_count": len(combo_results),
        })

results = pd.DataFrame(rows)
artifacts = pd.DataFrame(artifact_rows)
artifacts


In [ ]:
# Persist combined notebook outputs.
graph_manifest = output_dir / "index_manifest.json"
results_path = output_dir / "hh_mimic_subset_property_graph_results.csv"
artifacts_path = output_dir / "hh_property_graph_artifacts_manifest.csv"
results.to_csv(results_path, index=False)
artifacts.to_csv(artifacts_path, index=False)
print(graph_manifest)
print(results_path)
print(artifacts_path)
results[["id", "combo_slug", "model", "embedding_model", "gold_answer", "predicted"]]
